# **MODULO 1: Analissi de mercado musical para el artista**


* **Datos:**
    * Fuente de datos: Last.fm API (consumo real de usuarios)
    * Datos objetivo: Streams por país, Género musical, Playlist placement, TikTok trends, Crecimiento de artistas similares...
    * EXTRA DATOS: EDAD DE LA AUDIENCIA PARA AVERIGUAR TARGET SEGÚN PLATAFORMA. (MÉTRICAS AUDIENCIA TIK TOK)

* **Análisis (EDA):** crecimiento por género, países con más crecimiento, correlación entre playlist y streams, estacionalidad.
    * Feature Engineering: Ratio energía/valence, duración normalizada, explicit flag, features por artista

* **ML:** Predicción de streams de una canción, Predicción de viralidad, 
    * Modelos: Random Forest, XGBoost, LSTM para series temporales

* Output: “Tu canción tiene 65% probabilidad de viralizarse en México”, “Este beat funciona mejor en playlist de chill trap”





### **ÍNDICE DE IMPRESCINDIBLES PARA MODULO 1**



* **Módulo es el core estratégico: entender el mercado + base para predecir hits.**


* **DATOS: Se usan los datos de Last.fm API ya que  se basan en consumo real de usuarios.**

### Imports

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
 


# **DATA**

## Imports

In [26]:
import time
import requests
import pandas as pd
import os 

## Recopilación de datos API

In [27]:

API_KEY = '63e059c3c912a3f642daf2372484d183'
BASE_URL = 'http://ws.audioscrobbler.com/2.0/'
DELAY = 0.5  


### Data base para el análisis:

* Función data TopTracks

In [28]:
def fetch_top_tracks(page=1, limit=200, retries=5):
    params = {
        'method': 'chart.getTopTracks',
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                wait_time = 2 ** attempt
                print(f"[{response.status_code}] Retry en {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            time.sleep(2 ** attempt)

    print(f"❌ Fallo total en página {page}")
    return None

* No repetir datos del csv que ya tenemos

In [29]:
if os.path.exists("backup_manual.csv"):
    df_existing = pd.read_csv("backup_manual.csv")
else:
    df_existing = pd.DataFrame()

In [30]:
# crea conjunto de canciones existentes:
existing_tracks = set()

if not df_existing.empty:
    for _, row in df_existing.iterrows():
        key = f"{row['name']}_{row['artist']}"
        existing_tracks.add(key)

* Batch & DataFrame para TopTracks

In [31]:
all_tracks = []

for page in range(1, 50):  # empieza pequeño

    data = fetch_top_tracks(page=page)

    if data is None:
        print(f"⚠️ Saltando página {page}")
        time.sleep(5)
        continue

    tracks = data.get('tracks', {}).get('track', [])

    for track in tracks:
        name = track.get('name')
        artist = track.get('artist', {}).get('name')

        key = f"{name}_{artist}"

        # Evitar duplicados
        if key in existing_tracks:
            continue

        all_tracks.append({
            'name': name,
            'artist': artist,
            'playcount': track.get('playcount'),
            'listeners': track.get('listeners'),
            'mbid': track.get('mbid'),
            'duration': track.get('duration'),
            'url': track.get('url')
        })

        existing_tracks.add(key)

    time.sleep(DELAY)

    # Guardado frecuente
    if page % 10 == 0:
        df_temp = pd.DataFrame(all_tracks)
        df_total = pd.concat([df_existing, df_temp], ignore_index=True)
        df_total.to_csv("backup_manual.csv", index=False)
        print(f"Guardado en página {page}")

[502] Retry en 1s...
Guardado en página 10
Error: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)
[502] Retry en 1s...
Guardado en página 20
[502] Retry en 1s...
Guardado en página 30
[502] Retry en 1s...
[502] Retry en 1s...
[502] Retry en 1s...
Guardado en página 40


* Guardado frecuente

In [32]:
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv("backup_manualt.csv", index=False)

print("Recolección completada")

Recolección completada


In [33]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 19799 entries, 0 to 19798
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        19799 non-null  str   
 1   duration    19799 non-null  object
 2   playcount   19799 non-null  object
 3   listeners   19799 non-null  object
 4   mbid        16040 non-null  str   
 5   url         19799 non-null  str   
 6   streamable  9999 non-null   str   
 7   artist      19799 non-null  str   
 8   image       9999 non-null   str   
dtypes: object(3), str(6)
memory usage: 1.4+ MB


#### Erroes API 23-03-2026:

* Errores en peticiones:

| Código | Significado | Quién tiene el problema     |
| ------ | ----------- | --------------------------- |
| 403    | Forbidden   | ❌ tú (API key, permisos)    |
| 429    | Rate limit  | ⚠️ tú (demasiadas requests) |
| 502    | Bad Gateway | ❌ servidor (Last.fm)        |

* Errores después de obtener data: 
    - Bug en menos de 10K datos
    - Solucionar problema y no volver a guardar data duplicada
    - Descarga de datos sigue siendo insuficiente. Tenemos menos de 20K y necesitamos 60K. (Error 2 en doc ERRORS -> Solución: Combinar fuentes).

* Solucionar ERROR 2: descarga de data de más generos y países para completar las filas faltantes con información de valores para el análisis.


### Data por países:

* Función data países:

In [34]:
def fetch_geo_tracks(country, page=1, limit=200, retries=5):
    params = {
        'method': 'geo.getTopTracks',
        'country': country,
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                time.sleep(2 ** attempt)
                continue

            response.raise_for_status()
            return response.json()

        except:
            time.sleep(2 ** attempt)

    return None

* Lista por paises:

In [ ]:
countries = ['Spain', 'United States', 'United Kingdom', 'Brazil', 'Germany', 'France', 'Mexico', 'Peru', 'Japan', 'Netherlands']

for country in countries:
    print(f"🌍 País: {country}")

    for page in range(1, 20):

        data = fetch_geo_tracks(country, page)

        if data is None:
            continue

        tracks = data.get('tracks', {}).get('track', [])

        for track in tracks:
            name = track.get('name')
            artist = track.get('artist', {}).get('name')

            key = f"{name}_{artist}"

            if key in existing_tracks:
                continue

            all_tracks.append({
                'name': name,
                'artist': artist,
                'playcount': track.get('playcount'),
                'listeners': track.get('listeners'),
                'mbid': track.get('mbid'),
                'duration': track.get('duration'),
                'url': track.get('url'),
                'country': country
            })

            existing_tracks.add(key)

        time.sleep(DELAY)

🌍 País: Spain
🌍 País: United States
🌍 País: United Kingdom
🌍 País: Brazil
🌍 País: Germany
🌍 País: France
🌍 País: Mexico


In [38]:
FILE_NAME = 'backup_manual.csv'
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv(FILE_NAME, index=False)

print("Datos de géneros guardados")

Datos de géneros guardados


### Data por género musical:

In [39]:
def fetch_tag_tracks(tag, page=1, limit=200, retries=5):
    params = {
        'method': 'tag.getTopTracks',
        'tag': tag,
        'api_key': API_KEY,
        'format': 'json',
        'page': page,
        'limit': limit
    }

    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, timeout=10)

            if response.status_code in [429, 500, 502, 503]:
                wait_time = 2 ** attempt
                print(f"[{response.status_code}] Retry en {wait_time}s...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()
            return response.json()

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            time.sleep(2 ** attempt)

    print(f"Fallo en tag {tag}, página {page}")
    return None

In [44]:
def get_top_tags():
    params = {
        'method': 'tag.getTopTags',
        'api_key': API_KEY,
        'format': 'json'
    }

    response = requests.get(BASE_URL, params=params)
    data = response.json()

    tags = [tag['name'] for tag in data['toptags']['tag']]
    return tags

tags = get_top_tags()
print(tags[:21])

['rock', 'electronic', 'seen live', 'alternative', 'pop', 'indie', 'female vocalists', 'metal', 'alternative rock', 'jazz', 'classic rock', 'ambient', 'experimental', 'folk', 'indie rock', 'punk', 'Hip-Hop', 'hard rock', 'black metal', 'instrumental', 'singer-songwriter']


In [54]:
tags = ['rock', 'electronic', 'seen live', 'alternative', 'pop', 'indie', 'female vocalists', 'metal', 'alternative rock', 'jazz', 'classic rock', 'ambient', 'experimental', 'folk', 'indie rock', 'punk', 'Hip-Hop', 'hard rock', 'black metal', 'instrumental', 'singer-songwriter']

for tag in tags:
    print(f"🎧 Género: {tag}")

    for page in range(1, 20):

        data = fetch_tag_tracks(tag, page)

        if data is None:
            continue

        tracks = data.get('tracks', {}).get('track', [])

        for track in tracks:
            name = track.get('name')
            artist = track.get('artist', {}).get('name')

            key = f"{name}_{artist}"

            # 🚫 evitar duplicados
            if key in existing_tracks:
                continue

            all_tracks.append({
                'name': name,
                'artist': artist,
                'playcount': track.get('playcount'),
                'listeners': track.get('listeners'),
                'mbid': track.get('mbid'),
                'duration': track.get('duration'),
                'url': track.get('url'),
                'country': 'TAG',
                'genre_tag': tag
            })

            existing_tracks.add(key)

        time.sleep(DELAY)

    print(f"Completado género: {tag}")

🎧 Género: rock
[502] Retry en 1s...
[502] Retry en 2s...
Completado género: rock
🎧 Género: electronic
[502] Retry en 1s...
[502] Retry en 2s...
[502] Retry en 4s...
[502] Retry en 1s...
Completado género: electronic
🎧 Género: seen live
[502] Retry en 1s...
Completado género: seen live
🎧 Género: alternative
[502] Retry en 1s...
Error: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)
Completado género: alternative
🎧 Género: pop
Completado género: pop
🎧 Género: indie
[502] Retry en 1s...
[502] Retry en 2s...
Completado género: indie
🎧 Género: female vocalists
Error: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read timed out. (read timeout=10)
Completado género: female vocalists
🎧 Género: metal
Completado género: metal
🎧 Género: alternative rock
Completado género: alternative rock
🎧 Género: jazz
[502] Retry en 1s...
Completado género: jazz
🎧 Género: classic rock
Error: HTTPConnectionPool(host='ws.audioscrobbler.com', port=80): Read

In [55]:
df_new = pd.DataFrame(all_tracks)
df_final = pd.concat([df_existing, df_new], ignore_index=True)

df_final.drop_duplicates(subset=['name', 'artist'], inplace=True)

df_final.to_csv(FILE_NAME, index=False)

print("Datos de géneros guardados")

Datos de géneros guardados


In [56]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 100228 entries, 0 to 100227
Data columns (total 11 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   name        100228 non-null  str   
 1   duration    100228 non-null  object
 2   playcount   19799 non-null   object
 3   listeners   25013 non-null   object
 4   mbid        89266 non-null   str   
 5   url         100228 non-null  str   
 6   streamable  9999 non-null    str   
 7   artist      100228 non-null  str   
 8   image       9999 non-null    str   
 9   country     80429 non-null   str   
 10  genre_tag   75215 non-null   str   
dtypes: object(3), str(8)
memory usage: 8.4+ MB


# **EDA**

1. **Análisis del mercado musical (descriptivo):** (1) Top canciones / artistas por periodo (tiempo); (2) Evolución temporal de popularidad; (3) Géneros en crecimiento vs decrecimiento

2. **Análisis por género:**

* Distribución de features por género: danceability, energy, valence, tempo

* ¿Qué hace único a cada género?

3. **Análisis geográfico** (1) Países con mayor consumo; (2) Diferencias culturales en features.

4. **Ciclos de éxito:** (1) Duración de popularidad de una canción; (2) Tiempo entre picos de streams; (3) Estacionalidad (ej: “hits de verano”)

**Features engineering (clave para predecir un hit):**


1. **Correlación con popularidad:** Qué variables impactan más: danceability ↑, energy ↑, valence (positivo vs triste)

* Heatmap de correlaciones

2. **Ingeniería de variables:** (1) Ratio energía/valence; (2) Duración normalizada; (3) Explicit vs no explicit; (4) Features agregadas por artista

3. **Segmentación de canciones:** 

* Clustering para KMeans
* Tipos de canciones: “club hit”, “chill”, “sad viral”,... 


### API Request

In [ ]:
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=chart.gettoptracks&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")
music = pd.DataFrame(ret.json()["tracks"]["track"])
music.head()
# Cleaning the data frame:
music.drop(['mbid', 'image', 'url'],axis=1)


### Filtrando ds:

* Nombre y duracion de las canciones
* Comparación de la canción por titulo (happy or sad)
* Duración promedio de los top30 tracks


#### Nombre y duración de las canciones


In [ ]:
for track in ret.json()["tracks"]["track"]:
  print(track["name"], ". Duracion:", track["duration"],"minutos")


#### Comparación de grupos (canciones 'happy'-'sad'):

In [ ]:
df_happy = music[music['name'].str.contains('happy',case=False,na=False)]
df_happy.count()
df_sad = music[music['name'].str.contains('sad', case=False, na=False)]
df_sad.count()


#### Duración promedio de top30 tracks:

In [ ]:
df_top30_duration = music.sort_values(by='playcount', ascending=False) # .head(30)
df_top30_duration.iloc[:30]
df_top30_duration.head()

df_top30_duration.loc[:, 'duration'] = pd.to_numeric(df_top30_duration['duration'], errors='coerce')
mean = df_top30_duration['duration'].mean()
mean


### Procedencia de los artisitas top10:
 
* Oyentes de artists top10

**Info api request:**

In [ ]:
ret_artist = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=geo.gettopartists&country=spain&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret_artist.json()
artists_country = pd.DataFrame(ret_artist.json()["topartists"]["artist"])
artists_country.head()


**Oyentes de los artistas top 10:**

In [3]:
df_top10 = music.head(10)
df_top10['duration'] = pd.to_numeric(df_top10['duration'], errors='coerce')
df_top10['listeners'] = pd.to_numeric(df_top10['listeners'], errors='coerce')

df_top10['artist_name'] = df_top10['artist'].apply(lambda x: x['name'])

plt.figure(figsize = (15, 5))

plt.bar(df_top10['artist_name'], df_top10['listeners'], color = ['skyblue'])
plt.xlabel('Artist name')
plt.ylabel('Listeners count')
plt.title("Listener of top 10 artist")
plt.show()


NameError: name 'music' is not defined

### Géneros musicales en tendencia global

In [ ]:

def get_trending_tags(api_key):
    url = f"https://ws.audioscrobbler.com/2.0/?method=tag.getTopTags&api_key={api_key}&format=json"
    response = requests.get(url)
    return response.json()['toptags']['tag']

api_key = '63e059c3c912a3f642daf2372484d183'
get_trending_tags(api_key)
ret_generes = get_trending_tags(api_key)
top_generes = pd.DataFrame(ret_generes)[['name', 'count', 'reach']]
top_generes.head()

# **ML**

1. **Definición del problema:** 

* Regresión → predecir popularidad (0–100)

* Clasificación → hit vs no hit

2. **Modelos básicos (mínimo viable):** (1) Linear Regression; (2) Random Forest; (3) XGBoost / LightGBM

3. **Evaluación:** (1) RMSE / MAE (regresión); (2) Accuracy / F1 (clasificación)

4. **Interpretabilidad:**

* Feature importance

* SHAP values (muy top para destacar)